# LIBs

In [1]:
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix ,ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
import numpy as np
from itertools import product
import joblib
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import SGDRegressor

import matplotlib.pyplot as plt
from tensorflow.keras import Sequential
import warnings
warnings.filterwarnings('ignore')


c:\Users\Mariam Mahmoud\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


# Func

#### Model Evaluation

In [2]:
def model_evaluation(y_true, y_pred):
    print("Accuracy:", accuracy_score(y_true, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


#### flatten fun


In [3]:
def flatten_data(x_train, x_test):
    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    flat_train = x_train.reshape(n_train, x_train[0].size)
    flat_test  = x_test.reshape(n_test,  x_test[0].size)

    return flat_train, flat_test

#### K-Fold

In [5]:
def k_fold_cross_validation(model, x_data, y_data):
    k = 5
    n = x_data.shape[0]
    base = n // k

    scores = []

    for i in range(k):
        left = i * base
        right = (i + 1) * base if i != k - 1 else n

        X_val = x_data[left:right]
        y_val = y_data[left:right]

        X_tr = np.concatenate((x_data[:left], x_data[right:]), axis=0)
        y_tr = np.concatenate((y_data[:left], y_data[right:]), axis=0)

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        scores.append(f1_score(y_val, preds, average="macro"))

    return float(np.mean(scores)), float(np.std(scores))

In [6]:
def search_logreg_params(X_train, y_train, C_grid=None, tol_grid=None):
    C_grid = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 1.0, 5.0]
    tol_grid = [1e-2, 5e-3, 1e-3, 5e-4, 1e-4]
    best = {"score": -1.0, "C": None, "tol": None}

    for C, tol in product(C_grid, tol_grid):
        clf = LogisticRegression(
            C=C,
            tol=tol,
            multi_class="multinomial",
            random_state=42
        )

        mean_f1, _ = k_fold_cross_validation(clf, X_train, y_train)

        if mean_f1 > best["score"]:
            best.update({"score": mean_f1, "C": C, "tol": tol})

    return best["C"], best["tol"], float(best["score"])

#### tune Neural Network hyperparameter

In [7]:

def grid_search_mlp_simple(x_train, y_train, x_test, y_test, val_split=0.1):
    num_classes = 10

    grid = {
        "epochs": [20,40,60],
        "n_layers": [2, 3],
        "layer_size": [128 , 256 , 512],
        "l2": [1e-3],
        "lr": [0.001, 0.0001, 0.0003],
        "opt": ["Momentum", "Nesterov", "RMSprop", "Adam"],
        "batch": [32, 64],
        "act": ["sigmoid", "tanh", "relu"],
    }

    def make_opt(name, lr):
        if name == "SGD":
            return tf.keras.optimizers.SGD(learning_rate=lr)
        if name == "Momentum":
            return tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
        if name == "Nesterov":
            return tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=True)
        if name == "RMSprop":
            return tf.keras.optimizers.RMSprop(learning_rate=lr)
        if name == "Adam":
            return tf.keras.optimizers.Adam(learning_rate=lr)

    def build_model(n_layers, layer_size, act, l2v, lr, opt_name):
        reg = tf.keras.regularizers.l2(l2v) 

        model = tf.keras.Sequential()
        model.add(tf.keras.layers.Flatten(input_shape=(28, 28)))

        for i in range(n_layers):
            model.add(tf.keras.layers.Dense(layer_size, activation=act, kernel_regularizer=reg))

        model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

        model.compile(
            optimizer=make_opt(opt_name, lr),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )
        return model

    best_acc, best_params = -1.0, None
    all_results = {}

    combos = product(
        grid["epochs"], grid["n_layers"], grid["layer_size"], grid["l2"],
        grid["lr"], grid["opt"], grid["batch"], grid["act"]
    )

    for epochs, n_layers, layer_size, l2v, lr, opt_name, batch, act in combos:
        params = {
            "epochs": epochs, "n_layers": n_layers, "layer_size": layer_size, "l2": l2v,
            "lr": lr, "opt": opt_name, "batch": batch, "act": act
        }
        key = (epochs, n_layers, layer_size, l2v, lr, opt_name, batch, act)

        tf.keras.backend.clear_session()
        model = build_model(n_layers, layer_size, act, l2v, lr, opt_name)

        h = model.fit(x_train, y_train,validation_split=val_split,epochs=epochs,batch_size=batch,verbose=0)
    
        val_acc = float(max(h.history["val_accuracy"]))
        val_loss = float(min(h.history["val_loss"])) 

        all_results[key] = {
            "params": params,
            "best_val_accuracy": val_acc,
            "best_val_loss": val_loss,
            "history": h.history,
        }

        if val_acc > best_acc:
            best_acc = val_acc
            best_params = params

    final_model = build_model(
        best_params["n_layers"], best_params["layer_size"], best_params["act"],
        best_params["l2"], best_params["lr"], best_params["opt"]
    )
    final_model.fit(
        x_train, y_train,
        epochs=best_params["epochs"],
        batch_size=best_params["batch"],
        verbose=0
    )
    test_loss, test_acc = final_model.evaluate(x_test, y_test, verbose=0)

    return best_params, best_acc, float(test_acc), final_model, all_results

In [8]:
def train_ova(X_train, y_train, params=None):
    params = {} if params is None else params
    models = []

    for c in range(10):
        y_bin = (y_train == c).astype(np.float32)
        m = SGDRegressor(loss="squared_error", learning_rate="constant",
                         random_state=42, **params)
        m.fit(X_train, y_bin)
        models.append(m)

    return models


def predict_ova(models, X):
    preds = []

    for m in models:
        preds.append(m.predict(X))

    preds = np.column_stack(preds)
    return np.argmax(preds, axis=1)



In [19]:
def kfold_linear(X, y, param):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    acc, f1, precision, recall = [], [], [], []

    splits = list(cv.split(X, y)) 

    for i in range(5):
        train_idx, val_idx = splits[i]
        X_tr = X[train_idx]
        X_va = X[val_idx]
        y_tr = y[train_idx]
        y_va = y[val_idx]

        models = train_ova(X_tr, y_tr, param)
        y_hat = predict_ova(models, X_va)

        acc.append(accuracy_score(y_va, y_hat))
        f1.append(f1_score(y_va, y_hat, average="macro"))
        precision.append(precision_score(y_va, y_hat, average="macro"))
        recall.append(recall_score(y_va, y_hat, average="macro"))

    return {
        "acc_mean": np.mean(acc), "acc_std": np.std(acc),
        "f1_mean": np.mean(f1),   "f1_std": np.std(f1),
        "precision_mean": np.mean(precision),   "precision_std": np.std(precision),
        "recall_mean": np.mean(recall),   "recall_std": np.std(recall),
    }

In [10]:
def tune_linear_hyperparameter(X, y):
    best_f1 = -1
    best_params = None
    best_mean = None
    for eta0 in [0.0001,0.0003,0.001,0.003,0.01,0.03,0.1,0.3]:
        for max_iter in  [1000, 5000, 10000]:
            for tol in [1e-3, 1e-4, 1e-5 , 1e-6]:

                params = {
                    "eta0": eta0,
                    "max_iter": max_iter,
                    "tol": tol
                }

                kfold = kfold_linear(X, y, params)
                if kfold["f1_mean"] > best_f1:
                    best_f1 = kfold["f1_mean"]
                    best_params = params
                    best_mean = kfold

    return best_params, best_mean


## New

In [48]:
def kfold_cross_validation(X, y, params=None, model_type="logreg"):
    
    k = 5
    n = X.shape[0]
    fold_size = n // k
    
    acc_list = []
    f1_list = []
    precision_list = []
    recall_list = []
    
    for i in range(k):
        val_start = i * fold_size
        val_end = (i + 1) * fold_size if i != k - 1 else n
        
        X_val = X[val_start:val_end]
        y_val = y[val_start:val_end]
        
        X_train = np.concatenate([X[:val_start], X[val_end:]], axis=0)
        y_train = np.concatenate([y[:val_start], y[val_end:]], axis=0)
        
        if model_type == "logreg":
            thetas = train_ova_logreg(X_train, y_train, **params)
            y_pred = predict_ova_logreg(thetas, X_val)
        
        elif model_type == "linear":
            thetas = train_linear_ova(X_train, y_train, **params)
            y_pred = predict_linear_ova(X_val, thetas)
        
        elif model_type == "naive":
            model = GaussianNB()
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
        
        else:
            raise ValueError("Unknown model_type. Use 'logreg', 'linear', or 'naive'.")
        
        acc_list.append(accuracy_score(y_val, y_pred))
        f1_list.append(f1_score(y_val, y_pred, average="macro"))
        precision_list.append(precision_score(y_val, y_pred, average="macro"))
        recall_list.append(recall_score(y_val, y_pred, average="macro"))
    
    return {
        "acc_mean": np.mean(acc_list), "acc_std": np.std(acc_list),
        "f1_mean": np.mean(f1_list), "f1_std": np.std(f1_list),
        "precision_mean": np.mean(precision_list), "precision_std": np.std(precision_list),
        "recall_mean": np.mean(recall_list), "recall_std": np.std(recall_list),
    }

In [49]:
def tune_hyperparameters(X, y, model_type=""):
    best_f1 = -1
    best_params = None
    best_result = None
    
    learning_rate = [0.01, 0.001 ,0.1]
    epochs = [1000,1500]
    tol = [1e-5, 1e-3, 1e-4]
    l2= [0.01,0.001]
    
    for lr, ep, t, l2_val in product(learning_rate, epochs, tol, l2):
        params = {
            "learning_rate": lr,
            "epochs": ep,
            "tol": t,
            "l2_lambda": l2_val
        }
        
        kfold_result = kfold_cross_validation(X, y, params, model_type=model_type)
        
        if kfold_result["f1_mean"] > best_f1:
            best_f1 = kfold_result["f1_mean"]
            best_params = params
            best_result = kfold_result
            print(f"New best F1: {best_f1:.4f} | LR={lr}, Epochs={ep}, Tol={t}, L2={l2_val}")
    
    return best_params, best_result

# DATA

In [4]:

data_dir = r"D:\machine project\10000"  
X = []
y = []
target_size = (28, 28)
for label in range(10):
    path = os.path.join(data_dir, str(label))
    
    for name in os.listdir(path): 
        
        img_path = os.path.join(path, name)
        img = Image.open(img_path).convert("L") 
        img = img.resize(target_size)
        img_array = np.array(img, dtype="float32") 
        
        X.append(img_array)
        y.append(label)

X = np.array(X, dtype="float32") /255.0 #normalization
y = np.array(y, dtype="int64")

print("X shape:", X.shape) 
print("y shape:", y.shape)


X shape: (10000, 28, 28)
y shape: (10000,)


# Models

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,stratify=y, random_state=42)

## Logistic Classification

#### tune hyperparameter 

In [13]:
X_train_lr, X_test_lr = flatten_data(X_train, X_test)
best_c, best_tol, best_acc = search_logreg_params(X_train_lr, y_train)

#### training model

In [14]:
log_reg = LogisticRegression(C=best_c,tol=best_tol,multi_class='multinomial',random_state=42)
log_reg.fit(X_train_lr, y_train)


,penalty,'l2'
,dual,False
,tol,0.001
,C,0.1
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'multinomial'


#### testing model

In [15]:
y_pred = log_reg.predict(X_test_lr)
model_evaluation(y_test,y_pred)
f1 = f1_score(y_test, y_pred, average="macro")
print("F1 :", f1)

Accuracy: 0.772

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.81      0.80       200
           1       0.77      0.82      0.79       200
           2       0.83      0.73      0.78       200
           3       0.78      0.77      0.77       200
           4       0.74      0.76      0.75       200
           5       0.79      0.80      0.79       200
           6       0.83      0.86      0.84       200
           7       0.72      0.74      0.73       200
           8       0.76      0.66      0.71       200
           9       0.73      0.78      0.75       200

    accuracy                           0.77      2000
   macro avg       0.77      0.77      0.77      2000
weighted avg       0.77      0.77      0.77      2000


Confusion Matrix:
[[162   9   1   1   7   4   5   0   3   8]
 [  3 164   4   7   1   3   1  11   6   0]
 [  4   6 147  10   6   3   2  10   3   9]
 [  9   3   6 153   8   4   0   4   9   4]
 [  3   7 

## NEW LOgistic

In [51]:
X_train_lr, X_test_lr = flatten_data(X_train, X_test)
best_params, best_result = tune_hyperparameters(X_train_lr, y_train, "logreg")
print("Best params:", best_params)
print("F1 mean:", best_result["f1_mean"])
print("F1 std:", best_result["f1_std"])

New best F1: 0.6617 | LR=0.01, Epochs=1000, Tol=1e-05, L2=0.01
New best F1: 0.6756 | LR=0.01, Epochs=1500, Tol=1e-05, L2=0.01


KeyboardInterrupt: 

In [ ]:
def add_bias(X):
    m = X.shape[0]
    return np.hstack([np.ones((m, 1)), X])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train_ova_logreg(X, y, learning_rate=0.1, tol=1e-5, epochs=1000, l2_lambda=0.01, num_classes=10):

    X = add_bias(X)
    m, n = X.shape  #  X shape: (10000, 784) , m = num of samples(imgs), n = num of features

    thetas = np.zeros((num_classes, n))  # كل صف θ لفئة واحدة
  
    for c in range(num_classes): # loop over each class (numbers 0-9)
        y_bin = (y == c).astype(np.float32)  # if y(label) == c then 1 else 0 
        theta = np.zeros(n)  # theta for every pixel/feature for each image     
        for epoch  in range(epochs):
            z = X.dot(theta)
            h = sigmoid(z)
            gradient = (1/m) * X.T.dot(h - y_bin)
            gradient += (l2_lambda / m) * theta
            theta_new = theta - learning_rate * gradient
            
            # Check convergence
            if np.linalg.norm(theta_new - theta, ord=1) < tol:
                break
            theta = theta_new
            
        thetas[c] = theta
    return thetas

def predict_ova_logreg(thetas, X):
    X = add_bias(X)
    z = X.dot(thetas.T)  # shape (num_samples, num_classes)
    h = sigmoid(z)
    return np.argmax(h, axis=1)


In [44]:
thetas = train_ova_logreg(X_train_lr, y_train)
y_pred = predict_ova_logreg(thetas, X_test_lr)

model_evaluation(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")
print("F1 :", f1)


Accuracy: 0.7415

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.79      0.79       200
           1       0.68      0.79      0.73       200
           2       0.82      0.73      0.77       200
           3       0.78      0.78      0.78       200
           4       0.69      0.76      0.72       200
           5       0.77      0.72      0.75       200
           6       0.76      0.84      0.80       200
           7       0.71      0.68      0.69       200
           8       0.72      0.57      0.64       200
           9       0.73      0.77      0.75       200

    accuracy                           0.74      2000
   macro avg       0.74      0.74      0.74      2000
weighted avg       0.74      0.74      0.74      2000


Confusion Matrix:
[[158  12   0   3   9   4   6   0   3   5]
 [  3 157   5   6   2   3   5  10   7   2]
 [  2   7 146  10   7   1   2   8   6  11]
 [  7   5   8 155  10   2   0   3   5   5]
 [  4  10

## softmax logistic

In [23]:
def add_bias(X):
    m = X.shape[0]
    return np.hstack([np.ones((m, 1)), X])

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)  # numerical stability
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def train_softmax_logreg(
    X, y,
    learning_rate=0.01,
    tol=1e-5,
    epochs=500,
    l2_lambda=0.001,
    num_classes=10
):
    X = add_bias(X)
    m, n = X.shape

    # One-hot encoding
    Y = np.zeros((m, num_classes))
    Y[np.arange(m), y] = 1

    # weights
    W = np.zeros((n, num_classes))

    for epoch in range(epochs):
        scores = X @ W                    # (m, num_classes)
        probs = softmax(scores)

        gradient = (1/m) * X.T @ (probs - Y)
        gradient += (l2_lambda / m) * W   # L2 regularization

        W_new = W - learning_rate * gradient

        if np.linalg.norm(W_new - W, ord=1) < tol:
            break

        W = W_new

    return W


In [21]:
def predict_softmax_logreg(W, X):
    X = add_bias(X)
    scores = X @ W
    probs = softmax(scores)
    return np.argmax(probs, axis=1)


In [25]:
thetas = train_softmax_logreg(X_train_lr, y_train)
y_pred = predict_softmax_logreg(thetas, X_test_lr)

model_evaluation(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")
print("F1 :", f1)

Accuracy: 0.6655

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.77      0.73       200
           1       0.52      0.78      0.62       200
           2       0.86      0.58      0.70       200
           3       0.72      0.71      0.72       200
           4       0.59      0.63      0.61       200
           5       0.77      0.71      0.74       200
           6       0.66      0.79      0.72       200
           7       0.63      0.61      0.62       200
           8       0.73      0.42      0.54       200
           9       0.64      0.65      0.65       200

    accuracy                           0.67      2000
   macro avg       0.68      0.67      0.66      2000
weighted avg       0.68      0.67      0.66      2000


Confusion Matrix:
[[153  13   0   4  12   3   6   1   1   7]
 [  1 155   2   6   8   1  13   8   1   5]
 [  4  16 117  10  10   4   8  11   5  15]
 [ 11   8  11 143  12   2   0   2   5   6]
 [ 16  25

## Deep Learning

In [24]:
params, val_acc, test_acc, best_model, all_results = grid_search_mlp_simple(X_train, y_train, X_test, y_test, val_split=0.1)


[1/1 | models 100.0% | epochs 100.0%] run    9.6s | elapsed 00:00:10 | ETA 00:00:00 | best_val_acc 0.2675


In [16]:

model3 = Sequential()
#### Input layer
model3.add(tf.keras.layers.Flatten(input_shape=(28,28)))
#### Hidden layer
model3.add(tf.keras.layers.Dense(512, activation='relu',kernel_regularizer=tf.keras.regularizers.l2(0.001)))
model3.add(tf.keras.layers.Dense(512, activation='relu',kernel_regularizer=tf.keras.regularizers.l2(0.001)))
model3.add(tf.keras.layers.Dense(512, activation='relu',kernel_regularizer=tf.keras.regularizers.l2(0.001)))
#### Output layer
model3.add(tf.keras.layers.Dense(10,activation=tf.keras.activations.softmax)) 
model3.summary()
model3.compile(
    optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.0003),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model3.fit(
    X_train, y_train,
    epochs=600,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    shuffle=True
)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       401,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 932,362 (3.56 MB)

 Trainable params: 932,362 (3.56 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/600
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.3110 - loss: 3.2819 - val_accuracy: 0.5325 - val_loss: 2.5412
Epoch 2/600
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.4981 - loss: 2.4386 - val_accuracy: 0.4538 - val_loss: 2.3808
Epoch 3/600
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5751 - loss: 2.0600 - val_accuracy: 0.6025 - val_loss: 1.8941
Epoch 4/600
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6124 - loss: 1.8490 - val_accuracy: 0.6150 - val_loss: 1.7947
Epoch 5/600
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6440 - loss: 1.6965 - val_accuracy: 0.6963 - val_loss: 1.5687
Epoch 6/600
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6628 - loss: 1.5987 - val_accuracy: 0.7212 - val_loss: 1.4538
Epoch 7/600
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6765 - loss: 1.5198 - val_accuracy: 0.6500 - val_loss: 1.6054
Epoch 8/600
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7018 - loss: 1.4349 - val_accu

In [17]:
model3.evaluate(X_test, y_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8850 - loss: 0.6785


[0.6784891486167908, 0.8849999904632568]

In [14]:
import csv

csv_path = "grid_summary.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["best_val_accuracy","best_val_loss","epochs","n_layers","layer_size","l2","lr","opt","batch","act"])
    
    for r in all_results.values():
        p = r["params"]
        writer.writerow([
            r["best_val_accuracy"],
            r["best_val_loss"],
            p["epochs"], p["n_layers"], p["layer_size"], p["l2"],
            p["lr"], p["opt"], p["batch"], p["act"]
        ])

print("Saved CSV to:", csv_path)


Saved CSV to: grid_summary.csv


## Linear

In [20]:
X_train_flat,X_test_flat=flatten_data(X_train, X_test)
best_params, best_cv = tune_linear_hyperparameter(X_train_flat, y_train)


In [21]:
best_cv


{'acc_mean': np.float64(0.7160000000000001),
 'acc_std': np.float64(0.005500000000000033),
 'f1_mean': np.float64(0.7134856845409441),
 'f1_std': np.float64(0.005462727026872431),
 'precision_mean': np.float64(0.7233199584045946),
 'precision_std': np.float64(0.005355501550182996),
 'recall_mean': np.float64(0.716),
 'recall_std': np.float64(0.005499999999999972)}

In [22]:
best_params

{'eta0': 0.0001, 'max_iter': 1000, 'tol': 1e-05}

In [23]:
best_params={'eta0': 0.0001, 'max_iter': 1000, 'tol': 1e-05}

In [24]:
models = train_ova(X_train_flat, y_train, best_params)

In [25]:

y_pred = predict_ova(models, X_test_flat)

In [26]:


print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred))



Accuracy: 0.7165
F1: 0.7146727249313407
              precision    recall  f1-score   support

           0       0.76      0.79      0.77       200
           1       0.76      0.73      0.75       200
           2       0.72      0.78      0.75       200
           3       0.78      0.71      0.74       200
           4       0.59      0.83      0.69       200
           5       0.92      0.54      0.68       200
           6       0.60      0.90      0.72       200
           7       0.79      0.61      0.69       200
           8       0.73      0.56      0.63       200
           9       0.73      0.71      0.72       200

    accuracy                           0.72      2000
   macro avg       0.74      0.72      0.71      2000
weighted avg       0.74      0.72      0.71      2000



## Naive Bayes

In [27]:
X_train_nv, X_test_nv = flatten_data(X_train, X_test)

nv_model = GaussianNB()
nv_model.fit(X_train_nv, y_train)

y_pred_nb = nv_model.predict(X_test_nv)
model_evaluation( y_test, y_pred_nb)
acc_nb = k_fold_cross_validation(GaussianNB(), X_train_nv, y_train)
print("Mean Accuracy:",acc_nb[0])

Accuracy: 0.5115

Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.59      0.58       200
           1       0.32      0.67      0.43       200
           2       0.57      0.35      0.43       200
           3       0.49      0.61      0.54       200
           4       0.54      0.43      0.48       200
           5       0.64      0.45      0.53       200
           6       0.68      0.58      0.63       200
           7       0.57      0.50      0.53       200
           8       0.47      0.39      0.42       200
           9       0.60      0.56      0.58       200

    accuracy                           0.51      2000
   macro avg       0.54      0.51      0.51      2000
weighted avg       0.54      0.51      0.51      2000


Confusion Matrix:
[[118  26   3   8  15   7  11   2   2   8]
 [  2 133   2  19   6   5   7  20   4   2]
 [  4  40  70  42   6   1   1  11  16   9]
 [ 11  25  11 122   7   2   0   5   6  11]
 [ 18  28

## Save Models

In [45]:
#joblib.dump(log_reg, "log_reg.joblib")
joblib.dump(nv_model, "nv_model.joblib")    #NV
#model3.save("best_model.keras")   #DL

#DL_model = tf.keras.models.load_model("best_model.keras")  #DL
#log_reg_loaded = joblib.load("log_reg.joblib")
#nv_model_loaded = joblib.load("nv_model.joblib")  

['nv_model.joblib']